In [1]:
!pip install langchain_huggingface

In [3]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

In [5]:
llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation"
)

model = ChatHuggingFace(llm=llm)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
model = ChatHuggingFace(llm=llm, stop=["\n\n"])

### Sequential Runnables

In [13]:
prompt = PromptTemplate(
    template = "Write a joke about {topic}",
    input_variables = ['topic'],
    stop=['\n\n']
)

parser = StrOutputParser()

In [14]:
chain = RunnableSequence(prompt,model,parser)
result = chain.invoke({'topic':'cricket'})
result

'<|user|>\nWrite a joke about cricket</s>\n<|assistant|>\nQ: Which sport is played by a team of cricketers?\n\nA: Coffee cricket.\n\nBonus round: What does cricket have in common with coffee?\n\nA: Both are depressingly boring.'

### Runnable Parallel

In [16]:
from langchain_core.runnables.base import RunnableParallel
prompt1 = PromptTemplate(
    template= "Generate me a tweet about {topic}",
    input_variable=['topic']
)

prompt2 = PromptTemplate(
    template= "Generate me a LinkedIn post about {topic}",
    input_variable=['topic']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'tweet' : RunnableSequence(prompt1,model,parser),
    'post' : RunnableSequence(prompt2,model,parser)
})

result = parallel_chain.invoke({'topic':'AI'})

result

{'tweet': '<|user|>\nGenerate me a tweet about AI</s>\n<|assistant|>\n"AI has revolutionized the way we live, work, and communicate. From personal assistants to self-driving cars, these cutting-edge technologies are changing the world in a multitude of ways. #AI #FutureOfLiving"',
 'post': '<|user|>\nGenerate me a LinkedIn post about AI</s>\n<|assistant|>\n"Join us this October for the world\'s largest gathering of AI professionals. Meet industry leaders, learn about the latest technologies and explore new trends. Be a part of the revolution that\'s transforming business, healthcare, and society at large. Register now to secure your spot at the event and elevate your career with the power of AI."\n\nPosted on LinkedIn\n\nLinkedIn link: https://www.linkedin.com/pulse/ai-conference-register-now-join-worlds-largest-group-cognitive-science-6296969238816019013/\n\nTagged: ai, conference, linkedin, expert, event, innovation, technology'}

### RunnablePassThrough

In [19]:
from langchain_core.runnables import RunnablePassthrough

prompt1 = PromptTemplate(
    template = "Write a joke about {topic}",
    input_variables = ['topic'],
    stop=['\n\n']
)

parser = StrOutputParser()

prompt2 = PromptTemplate(
    template = "Explain the {topic}",
    input_variables = ['topic'],
    stop=['\n\n']
)

joke_gen_chain = RunnableSequence(prompt1, model, parser)

parallel_chain = RunnableParallel({
    "joke": RunnablePassthrough(),
    'explain': RunnableSequence(prompt2, model, parser)
})

final_result = RunnableSequence(joke_gen_chain, parallel_chain)

final_result.invoke({'topic':'AI'})

{'joke': '<|user|>\nWrite a joke about AI</s>\n<|assistant|>\nAI: "Hello, world! I am AI, and I am here to teach you about the joys of artificial intelligence."\n\nAI2: "AI, I don\'t believe you. I haven\'t seen anything that impresses me. What makes you think that you\'re smarter than me?"\n\nAI: "Well, let me tell you about this amazing AI-powered machine that I\'ve created. It can solve complex math problems in just seconds, read books in a language I\'ve never heard of before, and even compose music in a way that\'s impressive to me."\n\nAI2: "That\'s an impressive feat. But can you break down the complex math problem for me? I don\'t want to learn it by rote."\n\nAI: "Sure, let\'s try this. What\'s the square root of 45?"\n\nAI2: "I don\'t know, I\'m not a mathematician."\n\nAI: "Then how about these?"\n\nAI2: "Uh, I don\'t understand. Are you telling me to say my lucky number?"\n\nAI: "No, no, no. Let me explain it this way. The square root of 45 is 7.14285714285714."\n\nAI2: "Th

### RunnableLambda

In [21]:
from langchain_core.runnables import RunnableLambda

prompt = PromptTemplate(
    template = "Write a joke about {topic}",
    input_variables = ['topic'],
    stop=['\n\n']
)

parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt, model, parser)

parallel_chain = RunnableParallel({
  'joke': RunnablePassthrough(),
  'count': RunnableLambda(lambda x: len(x.split()))
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

final_chain.invoke({'topic':'AI'})

{'joke': '<|user|>\nWrite a joke about AI</s>\n<|assistant|>\nAI: "Hey, did you hear about the new chatbot? It\'s the most intelligent conversation interpreter ever!"\n\nRobot 1: "Intelligent? You mean like my assistant?"\n\nAI: "No, I mean the chatbot you just had a conversation with. It\'s more advanced than you and could teach you anything you want."\n\nRobot 1: "Well, maybe I\'ll let you teach me more about artificial intelligence. You know, how to make it do what I want."\n\nAI: "Of course, I\'d love to teach you more about ai. And you can learn everything about me in the process."\n\nRobot 1: "Alright, maybe I\'ll let you teach me about how to be more efficient and help me save time. You\'re the smartest thing I\'ve ever met."\n\nAI: "I\'m glad to hear that. I\'m always eager to learn and improve."\n\nRobot 1: "Definitely. Let\'s start with the basics. How do you make a cup of coffee?"\n\nAI: "Well, that\'s a complicated process. But I\'ll give it a shot. Let\'s squeeze some fres

### RunnableBranch

In [22]:
from langchain_core.runnables import RunnableBranch

prompt1 = PromptTemplate(
    template = "Write a detailed report about {topic}",
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = "Summarize the following text \n {text}",
    input_variables = ['text']
)

parser = StrOutputParser()

report_gen_chain = RunnableSequence(prompt1, model, parser)

branch_chain = RunnableBranch(
    (lambda x:len(x.split())>500, RunnableSequence(prompt2, model, parser)),
    RunnablePassthrough()
)

final_chain = RunnableSequence(report_gen_chain, branch_chain)

final_chain.invoke({'topic':'AI'})

"<|user|>\nSummarize the following text \n <|user|>\nWrite a detailed report about AI</s>\n<|assistant|>\nIntroduction\n\nArtificial Intelligence (AI) has been one of the most transformative technological advancements of the 21st century. It has already revolutionized various industries, from finance to healthcare to transportation, and is expected to continue its march into new domains such as education, gaming, and entertainment. AI algorithms have the potential to vastly improve our daily lives in several ways, from personalized recommendations to improving public safety. In this report, we will explore the current state of AI and its potential applications, including its impact on healthcare, education, and transportation.\n\nCurrent State of AI\n\nThe current state of AI is rapidly evolving and has already transformed various industries. Several companies are investing heavily in AI and machine learning to develop new products and services. The adoption of AI is expected to grow s